# KIdney dataset

Here we are creating dataset for kidney dataset


In [1]:
import os
from pathlib import Path

current_path = Path.cwd().resolve()
repository_root = next(
    path
    for path in (current_path, *current_path.parents)
    if (path / "pyproject.toml").is_file()
)
os.chdir(repository_root)

repository_root


PosixPath('/home/max/repositories/MSIAutoEncoderWrapper')

In [2]:
# load dataset
from IPython.display import display

from  msi_dataset_manager.exploration import DatasetExplorer

# REMARK: date i download DB is  12.08.2026 (DD, MM, YYYY) 
explorer = DatasetExplorer(
    source="metaspace",
    # Save the METASPACE dataset catalogue in this directory.
    cache_dir="assets/local/datasets/metaspace",
    # Set to True to request the catalogue again and replace the local file.
    refresh_cache=False,
)

# The same interface can be initialized for PRIDE:
# pride_explorer = DatasetExplorer(source="pride")


# About kidney

I choosed kidney as it is most countable dataset


In [3]:
# Replace the key with another enumerable entry from `available_filters`.
organism_values = explorer.get_available_values("organism_part")
display(organism_values.head(30))

# Typical additional inspections:
# display(explorer.get_available_values("organism_part").head(30))
# display(explorer.get_available_values("polarity"))
# display(explorer.get_available_values("analyzer_type").head(30))
# display(explorer.get_available_values("|ionisation_source").head(30))
# display(explorer.get_|available_values("maldi_matrix").head(30))


,value,label,count,variants
0,Kidney,Kidney,4436,"Kidney (4078), kidney (342), Kidney (14), kid..."
1,Brain,Brain,2143,"Brain (2108), brain (35)"
2,Cell Line,Cell Line,971,Cell Line (971)
3,Liver,Liver,945,"Liver (887), liver (51), LIVER (7)"
4,Root,Root,870,"Root (457), root (413)"
5,Lung,Lung,809,"Lung (700), lung (109)"
6,Whole organism,Whole organism,809,"Whole organism (774), whole organism (35)"
7,leaf,leaf,784,"leaf (622), Leaf (162)"
8,Breast,Breast,514,Breast (514)
9,N/A,N/A,425,"N/A (423), n/a (2)"


In [4]:
broad_filters = {
    # Biological metadata
    "organism": "Mouse",
    "organism_part": "Brain",
    "condition": ["Wildtype", "Wtype", "N/A"],

    # Acquisition metadata
    "polarity": "Negative",
    # "ionisation_source": "MALDI", - we can normalize this later 

    # Annotation filters
    # "status": "FINISHED",
    "annotation_fdr": 0.1,
    # "has_optical_image": True,
    "min_annotation_count": 1,

}

results = explorer.filter(broad_filters)

display(results)
print(f"Found {len(results)} datasets")


METASPACE discovery:   0%|          | 0/3 [00:00<?, ?stage/s]

Current operation:   0%|          | 0/1 [00:00<?, ?operation/s]

,dataset_id,name,source,project_accession,project_url,organisms,organism_parts,condition,growth_conditions,diseases,...,unannotated_pixel_count,annotated_pixel_fraction,annotation_fdr,spatial_annotation_count,spatial_annotation_database_count,spatial_stats_status,molecule_count,unique_molecule_count,unique_molecules,excluded
0,2026-07-22_18h49m21s,"Unwashed Brain - Section 13 (Negative, m/z 70 ...",metaspace,None,https://metaspace2020.eu/dataset/2026-07-22_18...,Mouse,Brain,N/A,,,...,None,None,0.1,None,None,None,None,None,,False
1,2026-07-22_18h47m13s,"Unwashed Brain - Section 11 (Negative, m/z 70 ...",metaspace,None,https://metaspace2020.eu/dataset/2026-07-22_18...,Mouse,Brain,N/A,,,...,None,None,0.1,None,None,None,None,None,,False
2,2026-07-08_21h20m18s,279_WTsaline_1_S2_SM_Neg_20260706_AQ,metaspace,None,https://metaspace2020.eu/dataset/2026-07-08_21...,Mus musculus (mouse),Brain,N/A,,,...,None,None,0.1,None,None,None,None,None,,False
3,2026-07-08_21h19m44s,470_WTcisplatin_1_S2_SM_Neg_20260706_AQ,metaspace,None,https://metaspace2020.eu/dataset/2026-07-08_21...,Mus musculus (mouse),Brain,N/A,,,...,None,None,0.1,None,None,None,None,None,,False
4,2026-07-08_21h16m42s,470_WTcisplatin_1_S2_SM_Neg_20260706_AQ_ML,metaspace,None,https://metaspace2020.eu/dataset/2026-07-08_21...,Mus musculus (mouse),Brain,N/A,,,...,None,None,0.1,None,None,None,None,None,,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
218,2017-09-23_11h29m21s,20170922_ADP_MB_3-DAN018_5x84_60x60,metaspace,None,https://metaspace2020.eu/dataset/2017-09-23_11...,Mus musculus (mouse),Brain,Wildtype,Caged,,...,None,None,0.1,None,None,None,None,None,,False
219,2018-01-18_14h55m11s,Mouse brain_CMBT_2_MK,metaspace,None,https://metaspace2020.eu/dataset/2018-01-18_14...,Mus musculus (mouse),Brain,Wildtype,N/A,,...,None,None,0.1,None,None,None,None,None,,False
220,2018-01-18_15h51m20s,Mouse brain_MBT_4_MK,metaspace,None,https://metaspace2020.eu/dataset/2018-01-18_15...,Mus musculus (mouse),Brain,Wildtype,N/A,,...,None,None,0.1,None,None,None,None,None,,False
221,2018-01-24_19h05m40s,DESIMousebrain_coronal,metaspace,None,https://metaspace2020.eu/dataset/2018-01-24_19...,Mus musculus (mouse),Brain,N/A,N/A,,...,None,None,0.1,None,None,None,None,None,,False


Found 223 datasets


Here we can see that proposals that have sense are between 

In [5]:
coverage = explorer.count_mz_range_coverage(
    lower_bounds=[100 * i for i in range(1, 5)],
    upper_bounds=[100 * i for i in range(7, 29)],
)

range_counts_matrix = coverage.pivot(
    index="lower_bound",
    columns="upper_bound",
    values="dataset_count",
)

# from 200 to 900 we obtain 30 datasets, it should be enough
range_counts_matrix

upper_bound,700.0,800.0,900.0,1000.0,1100.0,1200.0,1300.0,1400.0,1500.0,1600.0,...,1900.0,2000.0,2100.0,2200.0,2300.0,2400.0,2500.0,2600.0,2700.0,2800.0
lower_bound,,,,,,,,,,,,,,,,,,,,,
100.0,45,35,34,29,29,26,24,24,21,5,...,5,5,5,5,5,1,1,1,1,1
200.0,141,131,111,70,57,40,38,38,33,17,...,17,16,16,16,16,12,12,12,12,12
300.0,160,150,128,79,66,49,47,47,42,26,...,26,24,18,18,18,14,13,13,13,13
400.0,161,151,129,80,67,50,48,48,43,27,...,27,24,18,18,18,14,13,13,13,13


In [6]:
# this one severs for individual filtering
matching = explorer.select_mz_range(
    min_mz=300,
    max_mz=1900,
)
matching

,dataset_id,name,source,project_accession,project_url,organisms,organism_parts,condition,growth_conditions,diseases,...,unannotated_pixel_count,annotated_pixel_fraction,annotation_fdr,spatial_annotation_count,spatial_annotation_database_count,spatial_stats_status,molecule_count,unique_molecule_count,unique_molecules,excluded
0,2026-04-22_21h00m50s,granular_layer_mouse_brain_null_mz_shift_10_fr...,metaspace,None,https://metaspace2020.eu/dataset/2026-04-22_21...,Mus musculus (mouse),Brain,Wildtype,Caged,,...,None,None,0.1,None,None,None,None,None,,False
1,2025-03-19_16h20m57s,2025_03_17_MCF_NEG,metaspace,None,https://metaspace2020.eu/dataset/2025-03-19_16...,Mus musculus (mouse),Brain,N/A,,,...,None,None,0.1,None,None,None,None,None,,False
2,2025-03-19_16h15m47s,2025_03_17_BRK_NEG,metaspace,None,https://metaspace2020.eu/dataset/2025-03-19_16...,Mus musculus (mouse),Brain,N/A,,,...,None,None,0.1,None,None,None,None,None,,False
3,2025-01-29_16h26m40s,2025_01_15_lipidoscreen1bis_norharmane_neg,metaspace,None,https://metaspace2020.eu/dataset/2025-01-29_16...,Mus musculus (mouse),Brain,N/A,,,...,None,None,0.1,None,None,None,None,None,,False
4,2025-01-29_16h29m40s,2025_01_15_lipidoscreen1bis_dan_hcl_neg,metaspace,None,https://metaspace2020.eu/dataset/2025-01-29_16...,Mus musculus (mouse),Brain,N/A,,,...,None,None,0.1,None,None,None,None,None,,False
5,2025-01-29_16h28m17s,2025_01_15_lipidoscreen1bis_dan_lipido_neg,metaspace,None,https://metaspace2020.eu/dataset/2025-01-29_16...,Mus musculus (mouse),Brain,N/A,,,...,None,None,0.1,None,None,None,None,None,,False
6,2024-03-25_12h21m08s,20240207_neg_Lipid_LeuEnk_ALM1_w3_CLMC_1,metaspace,None,https://metaspace2020.eu/dataset/2024-03-25_12...,Mus musculus (mouse),Brain,Wtype,,,...,None,None,0.1,None,None,None,None,None,,False
7,2024-02-15_20h26m38s,PNNL05A_V6b_CLMCAFAMM_Lipids_885_10ppm,metaspace,None,https://metaspace2020.eu/dataset/2024-02-15_20...,Mus musculus (mouse),Brain,Wildtype,N/A,,...,None,None,0.1,None,None,None,None,None,,False
8,2024-02-15_20h25m33s,PNNL05A_V6b_CLMCAFAMM_Lipids_885_3ppm,metaspace,None,https://metaspace2020.eu/dataset/2024-02-15_20...,Mus musculus (mouse),Brain,Wildtype,N/A,,...,None,None,0.1,None,None,None,None,None,,False
9,2024-02-02_00h29m28s,PNNL05A_V6b_CLMCAFAMM_Lipids_885_5ppm,metaspace,None,https://metaspace2020.eu/dataset/2024-02-02_00...,Mus musculus (mouse),Brain,Wildtype,N/A,,...,None,None,0.1,None,None,None,None,None,,False


In [4]:
filters = {
    # Biological and acquisition metadata
    "organism": "Mouse",
    "organism_part": "Liver",
    "polarity": "Negative",
    "condition": "Wildtype",  # matched tolerantly; also groups "Wildtype", "wildtype", etc.
    "mz_min": 200,
    "mz_max": 1400,

    # Annotation filters and molecular statistics
    "annotation_fdr": 0.1,
    "min_annotation_count": 1,
    "include_molecule_stats": True,
    "include_spatial_annotation_stats": True, # REMARK: it filters unique values based on `annotation_fdr`
}

results_liver = explorer.filter(filters)


METASPACE discovery:   0%|          | 0/3 [00:00<?, ?stage/s]

Current operation:   0%|          | 0/1 [00:00<?, ?operation/s]

In [5]:
display(results_liver)
print(f"Accepted {len(results_liver)} datasets") 

,dataset_id,name,source,project_accession,project_url,organisms,organism_parts,condition,growth_conditions,diseases,...,unannotated_pixel_count,annotated_pixel_fraction,annotation_fdr,spatial_annotation_count,spatial_annotation_database_count,spatial_stats_status,molecule_count,unique_molecule_count,unique_molecules,excluded
0,2025-08-19_19h29m48s,Lipids3_top_control_nodiverter,metaspace,None,https://metaspace2020.eu/dataset/2025-08-19_19...,Mouse,Liver,Wildtype,,,...,0,1.0,0.1,319,2,complete,207,5,"C16H14O8-H, C17H18O7-H, C19H32O4-H, C22H21O11-...",False
1,2025-08-19_19h33m37s,Lipids4_top_Nh4F_diverter,metaspace,None,https://metaspace2020.eu/dataset/2025-08-19_19...,Mouse,Liver,Wildtype,,,...,0,1.0,0.1,59,2,complete,37,0,,False
2,2025-08-19_19h33m18s,Lipids4_bottom_control_nodiverter,metaspace,None,https://metaspace2020.eu/dataset/2025-08-19_19...,Mouse,Liver,Wildtype,,,...,0,1.0,0.1,158,2,complete,109,6,"C12H15NO5-H, C15H10O7S-H, C17H12ClF3N2O-H, C28...",False
3,2025-08-19_19h29m30s,Lipids3_bottom_Nh4F_diverter,metaspace,None,https://metaspace2020.eu/dataset/2025-08-19_19...,Mouse,Liver,Wildtype,,,...,0,1.0,0.1,261,2,complete,182,10,"C20H24O3-H, C28H44N2O8S-H, C29H38N5O9-H, C29H4...",False
4,2025-08-19_19h28m58s,Lipids2_top_NH4F_diverter,metaspace,None,https://metaspace2020.eu/dataset/2025-08-19_19...,Mouse,Liver,Wildtype,,,...,0,1.0,0.1,348,2,complete,225,12,"C11H20O-H, C13H22O2-H, C14H14N2-H, C14H26O6-H,...",False
5,2025-08-19_19h23m41s,Lipids1_bottom_NH4F_diverter,metaspace,None,https://metaspace2020.eu/dataset/2025-08-19_19...,Mouse,Liver,Wildtype,,,...,0,1.0,0.1,417,2,complete,262,44,"C12H15N5O3-H, C17H20N4S-H, C17H22N2O3-H, C18H2...",False
6,2025-08-19_19h27m06s,Lipids2_bottom_control_nodiverter,metaspace,None,https://metaspace2020.eu/dataset/2025-08-19_19...,Mouse,Liver,Wildtype,,,...,0,1.0,0.1,180,2,complete,120,1,C33H37N5O5-H,False
7,2025-08-19_19h26m22s,Lipids1_top_control_nodiverter,metaspace,None,https://metaspace2020.eu/dataset/2025-08-19_19...,Mouse,Liver,Wildtype,,,...,0,1.0,0.1,309,2,complete,216,9,"C15H20O3-H, C16H26N2O16P2-H, C19H27N5-H, C20H2...",False
8,2025-06-30_15h03m07s,Tissue5_top_NH4F,metaspace,None,https://metaspace2020.eu/dataset/2025-06-30_15...,Mouse,Liver,Wildtype,,,...,0,1.0,0.1,1182,2,complete,829,149,"C10H12N2O7-H, C10H12O3-H, C10H13N4O7P-H, C10H1...",False
9,2025-06-30_15h02m48s,Tissue4_top_NH4F,metaspace,None,https://metaspace2020.eu/dataset/2025-06-30_15...,Mouse,Liver,Wildtype,,,...,0,1.0,0.1,696,2,complete,468,23,"C10H11NO2-H, C10H18N2O4-H, C11H21N3O5-H, C11H6...",False


Accepted 18 datasets


In [10]:
## to gigabyte transfer 
results_liver['total_size_bytes'].sum() / 10**(9)

np.float64(1.316907151)

## 4. Export filters


In [11]:
output_path = Path(
    "data/liver_workspace/configs/datasets/liver"
)

exported = explorer.export_selection(
    output_path,
    sort_by="download_size_bytes",
    ascending = False
)

exported

{'filters': PosixPath('data/liver_workspace/configs/datasets/liver/filter.json'),
 'selection': PosixPath('data/liver_workspace/configs/datasets/liver/selection.json')}